<a href="https://colab.research.google.com/github/elliemci/agents/blob/main/agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG

## Required Libraries

In [ ]:
!pip install datasets langchain langchain-community faiss-cpu openai smolagents chromadb

In [2]:
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.8 MB/s eta 0:00:00


## Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/AgentsCourse

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/AgentsCourse


In [3]:
from datasets import load_dataset
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from smolagents import CodeAgent, Tool, HfApiModel

from langchain_community.embeddings import HuggingFaceEmbeddings

## Environment Varaibles

In [4]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get('huggingface_hub_access_token')

## Tools

### Database Retrieval

1. Load MedMCQA

2. Prepare data: extract questions, contexts, and answers

3. Chunk the data

4. Embed and store in a vectorstore

5. Create a retriever tool

MedMCQA is a large-scale Multiple-Choice Question Answering dataset which contains real Medical exam Question Answering data set [medmcqa](https://huggingface.co/datasets/medmcqa) hast the follwing data fields:
* id: a string question identifier for each example
* question: a string text
* opa: Option A
* opb: Option B
* opc: Option D
* cop: Correct option
* choice_type {"single", "multi"}: question type, "single"-choice contains a single option, and "multi"-choice question contains a combination of multiple options
* exp: expert's explanation of the answer
* subject_name: medical subject name of thequestion
* topic_name: medical topic name

In [ ]:
# load MedMCQA from Huggingface
dataset = load_dataset("medmcqa", split="train") # for speed use "train[:1000]"

In [6]:
dataset[0]

{'id': 'e9ad821a-c438-4965-9f77-760819dfa155',
 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma',
 'opa': 'Hyperplasia',
 'opb': 'Hyperophy',
 'opc': 'Atrophy',
 'opd': 'Dyplasia',
 'cop': 2,
 'choice_type': 'single',
 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950',
 'subject_name': 'Anatomy',
 'topic_name': 'Urinary tract'}

### Clean up NA data

In [20]:
# handling missing values by converting to pandas df and using isna()
import pandas as pd

df = pd.DataFrame(dataset)
df.isna().sum()

,0
id,0
question,0
opa,0
opb,0
opc,0
opd,0
cop,0
choice_type,0
exp,21953
subject_name,0


In [13]:
# drop rows with missing values in columns exp and topic_name
df = df.dropna(subset=["exp"])
print(df.isna().sum())
df = df.dropna(subset=["topic_name"])
print(df.isna().sum())

id                  0
question            0
opa                 0
opb                 0
opc                 0
opd                 0
cop                 0
choice_type         0
exp                 0
subject_name        0
topic_name      73792
dtype: int64
id              0
question        0
opa             0
opb             0
opc             0
opd             0
cop             0
choice_type     0
exp             0
subject_name    0
topic_name      0
dtype: int64


In [14]:
# convert no missin values dataframe back to datasest
dataset = dataset.from_pandas(df)

In [15]:
# combine question and explanation for context
docs = []

for item in dataset:
  content = f"Q: {item['question']}\nA) {item['opa']} B) {item['opb']} C) {item['opc']} D) {item['opd']}\nAnswer: {item['cop']}\nExplanation: {item.get('exp', '')}"
  docs.append(Document(page_content=content, metadata={"id": item['id']}))

### Medical Dataset Retrieval Tool

In [22]:
def get_correct_answer_text(ex):
    """
    Given an example from MedMCQA, extract the actual text(s) of the correct answer(s).
    Works for both 'single' and 'multi' choice_type.
    """
    options = {
        "1": ex.get("opa", "").strip(),
        "2": ex.get("opb", "").strip(),
        "3": ex.get("opc", "").strip(),
        "4": ex.get("opd", "").strip()
    }

    # get the text from the correct answer or answers
    correct_option_raw = str(ex.get("cop", "")).strip()
    correct_indices = [opt.strip() for opt in correct_option_raw.split(",") if opt.strip()]

    # Fallback: infer choice_type from number of correct answers
    declared_type = ex.get("choice_type", "single").strip().lower()
    inferred_type = "multi" if len(correct_indices) > 1 else "single"

    # Use whichever is more accurate
    choice_type = inferred_type if declared_type not in {"single", "multi"} else declared_type

    correct_texts = [options.get(idx, f"[Unknown Option {idx}]") for idx in correct_indices]

    return correct_texts if choice_type == "multi" else correct_texts[0]


In [23]:
ex = {
    "question": "What causes increased blood pressure?",
    "opa": "Low salt intake",
    "opb": "Dehydration",
    "opc": "High sodium levels",
    "opd": "Regular exercise",
    "cop": "2, 3",
    "choice_type": "multi",
    "exp": "High sodium levels cause water retention, increasing blood volume and pressure.",
    "subject_name": "Physiology",
    "topic_name": "Cardiovascular System"
}

print(get_correct_answer_text(ex))

['Dehydration', 'High sodium levels']


In [25]:
from smolagents import Tool
from datasets import load_dataset
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm import tqdm
import os, shutil

# Setup
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = []

# Utility to get correct answer text
def get_correct_answer_text(ex):
    options = {
        "1": ex.get("opa", "").strip(),
        "2": ex.get("opb", "").strip(),
        "3": ex.get("opc", "").strip(),
        "4": ex.get("opd", "").strip()
    }

    correct_option_raw = str(ex.get("cop", "")).strip()
    correct_indices = [opt.strip() for opt in correct_option_raw.split(",") if opt.strip()]

    # fallback or check on consistency
    declared_type = ex.get("choice_type", "single").strip().lower()
    inferred_type = "multi" if len(correct_indices) > 1 else "single"
    choice_type = inferred_type if declared_type not in {"single", "multi"} else declared_type

    correct_texts = [options.get(idx, f"[Unknown Option {idx}]") for idx in correct_indices]
    return correct_texts if choice_type == "multi" else correct_texts[0]

# Tool definition
class MedDataChromaRetrieverTool(Tool):
    name = "med_data_chroma_retrieve"
    description = "Retrieves correct answers from MedMCQA using semantic similarity (Chroma vector store)."
    inputs = {
        "query": {
            "type": "string",
            "description": "The medical question you want to search."
        }
    }
    output_type = "string"

    def __init__(self, persist_directory="medmcqa_chromadb"):
        self.is_initialized = False
        self.vectorstore = None
        self.persist_directory = persist_directory

        if os.path.exists(persist_directory):
            self.vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)
            self.is_initialized = True
        else:
            os.makedirs(self.persist_directory, exist_ok=True)
            self._build_vector_index()

    def _build_vector_index(self):
        med_dataset = load_dataset("medmcqa", split="train[:1000]")

        for ex in tqdm(med_dataset, desc="Building vector index for MedMCQA dataset"):
            correct_answer_text = get_correct_answer_text(ex)
            explanation = str(ex.get("exp", "") or "").strip()
            subject = str(ex.get("subject_name", "") or "").strip()
            topic = str(ex.get("topic_name", "") or "").strip()

            content = f"""Question: {ex['question']}
Correct Answer: {correct_answer_text}"""
            if explanation:
                content += f"\nExplanation: {explanation}"
            if subject and subject.upper() != "NA":
                content += f"\nSubject: {subject}"
            if topic and topic.upper() != "NA":
                content += f"\nTopic: {topic}"

            metadata = {
                "subject": subject,
                "topic": topic,
                "answer": correct_answer_text
            }

            split_docs = text_splitter.create_documents([content], metadatas=[metadata])
            docs.extend(split_docs)

        if os.path.exists(self.persist_directory):
            shutil.rmtree(self.persist_directory)

        self.vectorstore = Chroma.from_documents(
            docs,
            embedding=embedding_model,
            persist_directory=self.persist_directory
        )
        self.vectorstore.persist()
        self.is_initialized = True

    def forward(self, query: str):
        if not self.is_initialized:
            return "Vector store not initialized."

        results = self.vectorstore.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

# initialize the medical data retrieval tool
med_data_chroma_retrieve_tool = MedDataChromaRetrieverTool()


In [26]:
# initialize the Hugging Face model
model = HfApiModel()
# create a CodeAgent agent which can use the med data retrieval tool or execute code to solve the problem
agent = CodeAgent(tools = [med_data_chroma_retrieve_tool],
                  model = model,
                  # authorize 'requests' and 'bs4' in case the agent uses code to search the web
                  additional_authorized_imports=["requests", "bs4"])
# test the retrival tool
response = agent.run("What are common causes of flear and symptoms of rhumatoid arthritis?")
print(response)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What are common causes of flear and symptoms of rhumatoid arthritis?                                            │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  causes_of_rheumatoid_arthritis = med_data_chroma_retrieve(query="common causes of rheumatoid arthritis")         
  print("Causes of rheumatoid arthritis:", causes_of_rheumatoid_arthritis)                                         
                                                                                                                   
  symptoms_of_rheumatoid_arthritis = med_data_chroma_retrieve(query="symptoms of rheumatoid arthritis")            
  print("Symptoms of rheumatoid arthritis:", symptoms_of_rheumatoid_arthritis)                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Causes of rheumatoid arthritis: Q: Most common cause of scleritis -
A) Rheumatoid arthritis B) SLE C) Sjogren's Syndrome D) Behcet's Disease
Answer: 0
Explanation: Ans. is 'a' i.e., Rheumato.id arthritis Scleritiso Scleritis is an uncommon disorder which is 
characterized by cellular infiltration, destruction of collagen and vascular remodelling. Scleritis is usually a 
bilateral disease and occurs most frequently in women. It is associated with connective tissue diseases in 50% of 
cases. Rheumatoid arthritis is the most common association . Other important causes are PAN, SLE, Ankylosing 
spondylitis, Wegener's granulomatosis, dermatomyositis, Reiter's syndrome, Non- specific arteritis, Polychondritis 
and Gout.Clinical features of Scleritiso Patients complain of moderate to severe pain which is deep and boring in 
character and often wakes thepatient early in the morning. Ocular pain radiates to the jaw and temple. It is 
associated with localised or diffuse redness, mild to severe photophobia and lacrimation. Occasionally there occurs
diminution of vision.Complicationso Complications are common in necrotizing scleritis and include sclerosing 
keratitis, Keratolysis, Complicated cataract, Uveitis and secondary glaucoma. Rarely, scleritis may also cause 
retinal detachment and macular edema. These are due to spread of inflammation from sclera into the uveal tract.

Q: Most common cardiac involvement in Rheumatoid arthritis is
A) Conduction defects B) Pericarditis C) Myocarditis D) Infective Endocarditis
Answer: 1
Explanation: (B) Pericarditis # CARDIAC COMPLICATIONS OF RHEUMATOID ARTHRITIS> Pericarditis: Asymptomatic. 
One-third of patients with seropositive RA.> Myocarditis.> Endocarditis.> Conduction defects.> Coronary 
vasculitis.> Granulomatous aortitis.

Q: Which of the following is seen in Seropositive Rheumatoid Arthritis?
A) Multiple joints affected B) Symmetrical joint symptoms C) Morning stiffness, joint pain and swelling D) All of 
the above
Answer: 3
Explanation: (D) All of the above[?]Rheumatoid arthritis (RA) is a chronic inflammatory disease of unknown etiology
marked by a symmetric, peripheral polyarthritis.It is the most common form of chronic inflammatory arthritis & 
often results in joint damage and physical disability.Because it is a systemic disease, RA may result in a variety 
of extraarticular manifestations, including fatigue, subcutaneous nodules, lung involvement, pericarditis, 
peripheral neuropathy, vasculitis & hematologic abnormalities. Rheumatoid Arthritis Criteria (1987 revision, 
American Rheumatism Association):Morning stiffness (in/around joints, >1 hr before maximal improvement)Arthritis 
(swelling) of 3 or more joint areas (observed by physician)Symmetric arthritis (swelling, NOT bony 
overgrowth)Arthritis of Hand joints (wrists, MCPs or PIPs)Rheumatoid nodulesRheumatoid factor (serum)Radiographic 
changes (erosions and/or peri-articular osteopenia in hand/wrist joints)Requirements: >4 of the above 7 criteria. 
Criteria 1-4 must have been present for at least 6 weeks.Characteristic hand signs:Z-thumbs, Boutonniere, 
swan-necking, ulnar deviation, muscle wasting, subluxation of MCP, sometimes Carpal tunnel syndrome, Atlantoaxial 
subluxationExtra-articular features; Sjogren 'ds syndrome, Reynaud's, vasculitis. Nodules (firm but usually 
painless)Pathology: Auto-antibody production (70%IgM,30%IgG) against joint tissues.Investigations:Blood: RA factor 
positive. Rheumatoid factor is not diagnostic. Present in 70%, but also in general population. Rheumatoid positive 
disease has worse prognosis and more extensive deformity. Anti-CCP antibodies test similar to rheumatoid factor, | 
ESR + | CRP, normochromic-normocytic anaemia common.X-ray: look for nodules, soft-tissue swelling, osteopaenia, 
deformity, erosions.Treatment:Aim to reduce long-term deformity & LoF. Steroids induce remission, beware SE's long 
termDMARDs are mainstay. All can cause myelosuppression and rash plus: 

[Step 1: Duration 6.92 seconds| Input tokens: 2,090 | Output tokens: 141]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 10 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({ "Causes of rheumatoid arthritis": "Common causes include autoimmune response, genetic factors,    
  environmental triggers, and hormonal influences.",                                                               
                 "Symptoms of rheumatoid arthritis": "Common symptoms include joint pain and stiffness,            
  symmetrical joint involvement, joint swelling and redness, fatigue, low-grade fever, weight loss,                
  extra-articular manifestations, and rheumatoid nodules." })                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: {'Causes of rheumatoid arthritis': 'Common causes include autoimmune response, genetic factors,
environmental triggers, and hormonal influences.', 'Symptoms of rheumatoid arthritis': 'Common symptoms include 
joint pain and stiffness, symmetrical joint involvement, joint swelling and redness, fatigue, low-grade fever, 
weight loss, extra-articular manifestations, and rheumatoid nodules.'}

[Step 10: Duration 30.41 seconds| Input tokens: 120,274 | Output tokens: 1,970]

{'Causes of rheumatoid arthritis': 'Common causes include autoimmune response, genetic factors, environmental triggers, and hormonal influences.', 'Symptoms of rheumatoid arthritis': 'Common symptoms include joint pain and stiffness, symmetrical joint involvement, joint swelling and redness, fatigue, low-grade fever, weight loss, extra-articular manifestations, and rheumatoid nodules.'}


#### Medical data Search Tool

### Web Search

### External API